### 1. Load DataSet

In [1]:
import dagshub
dagshub.init(repo_owner='iamdebasishdas123', repo_name='YouTube-Mood-Tracker', mlflow=True)
import mlflow

Accessing as iamdebasishdas123

Initialized MLflow to track repo "iamdebasishdas123/YouTube-Mood-Tracker"

Repository iamdebasishdas123/YouTube-Mood-Tracker initialized!

c:\Users\Debasish Das\Desktop\sentiment_analysis\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import numpy as np
import pandas as pd
# Load the data
def load_data():
    df = pd.read_csv('https://raw.githubusercontent.com/Himanshu-1703/reddit-sentiment-analysis/refs/heads/main/data/reddit.csv')
    return df
df = load_data()
df.head()

,clean_comment,category
0,family mormon have never tried explain them t...,1
1,buddhism has very much lot compatible with chr...,1
2,seriously don say thing first all they won get...,-1
3,what you have learned yours and only yours wha...,0
4,for your own benefit you may want read living ...,1


### 2. Preprocessing of Data

In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


nltk.download('stopwords')
nltk.download('wordnet')

# Define the preprocessing function
def preprocess_comment(comment):
    # Convert to lowercase
    comment = comment.lower()

    # Remove trailing and leading whitespaces
    comment = comment.strip()

    # Remove newline characters
    comment = re.sub(r'\n', ' ', comment)

    # Remove non-alphanumeric characters, except punctuation
    comment = re.sub(r'[^A-Za-z0-9\s!?.,]', '', comment)

    # Remove stopwords but retain important ones for sentiment analysis
    stop_words = set(stopwords.words('english')) - {'not', 'but', 'however', 'no', 'yet'}
    comment = ' '.join([word for word in comment.split() if word not in stop_words])

    # Lemmatize the words
    lemmatizer = WordNetLemmatizer()
    comment = ' '.join([lemmatizer.lemmatize(word) for word in comment.split()])

    return comment

[nltk_data] Downloading package stopwords to C:\Users\Debasish
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Debasish
[nltk_data]     Das\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [9]:
def preprocess_data(df):
    df=df.dropna()
    df=df.drop_duplicates()
    df = df[~(df['clean_comment'].str.strip() == '')]
    df['clean_comment'] = df['clean_comment'].apply(preprocess_comment)
    return df

In [10]:
data=preprocess_data(df)

#### Experiments Embedding Model

In [11]:
# Set or create an experiment
mlflow.set_experiment("Embedding_Experiments")

<Experiment: artifact_location='mlflow-artifacts:/f8195b3b2d90485782486f3af3a211cc', creation_time=1783534617348, experiment_id='1', last_update_time=1783534617348, lifecycle_stage='active', name='Embedding_Experiments', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [14]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

# Define experiment parameters
ngram_ranges = [ (1,2), (1,3)]  # unigram, bigram, trigram
vectorizers = {
    'bow': CountVectorizer(),
    'tfidf': TfidfVectorizer(),
    # 'word2vec': None  # Will be handled separately
}
feature_sizes = [5000, 7000, 9000, 11000, 13000]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    data['clean_comment'], data['category'], test_size=0.2, random_state=42
)
y_train = y_train.map({-1: 0, 0: 1, 1: 2})
y_test = y_test.map({-1: 0, 0: 1, 1: 2})

# Run experiments
for ngram_range in ngram_ranges:
    for vectorizer_name, vectorizer in vectorizers.items():
        for feature_size in feature_sizes:
            run_name = f"{vectorizer_name}_ngram{ngram_range}_features{feature_size}"
            with mlflow.start_run(run_name=run_name):
                # Set experiment parameters
                mlflow.log_params({
                    'ngram_range': ngram_range,
                    'vectorizer': vectorizer_name,
                    'feature_size': feature_size
                })

                # Handle different vectorizers
                if vectorizer_name in ['bow', 'tfidf']:
                    # Configure vectorizer
                    vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=feature_size) \
                        if vectorizer_name == 'bow' \
                        else TfidfVectorizer(ngram_range=ngram_range, max_features=feature_size)
                    
                    # Transform data
                    X_train_vec = vectorizer.fit_transform(X_train)
                    X_test_vec = vectorizer.transform(X_test)
                
                # Word2Vec implementation
                elif vectorizer_name == 'word2vec':
                    # Train Word2Vec model
                    tokenized_comments = [comment.split() for comment in X_train]
                    w2v_model = Word2Vec(tokenized_comments, vector_size=100, window=5, min_count=5, workers=4)
                    
                    # Create sentence vectors
                    def average_word_vectors(texts, model):
                        return np.array([
                            np.mean([model.wv[word] for word in text.split() if word in model.wv], axis=0)
                            for text in texts
                        ])
                    
                    X_train_vec = average_word_vectors(X_train, w2v_model)
                    X_test_vec = average_word_vectors(X_test, w2v_model)
                    
                # Train XGBoost model
                model = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
                model.fit(X_train_vec, y_train)
                
                # Evaluate model with accuracy, precision, recall, and F1 score
                y_pred = model.predict(X_test_vec)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred, average='weighted')
                recall = recall_score(y_test, y_pred, average='weighted')
                f1 = f1_score(y_test, y_pred, average='weighted')
                
                
                # Log metrics
                mlflow.log_metrics({
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1_score': f1
                })
                
                print(f"Completed experiment: {vectorizer_name}, ngram={ngram_range}, features={feature_size}, Accuracy={accuracy:.4f}")

Completed experiment: bow, ngram=(1, 2), features=5000, Accuracy=0.7466
🏃 View run bow_ngram(1, 2)_features5000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1/runs/e108695b8ee74c908cc6c0f1dccd69db
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1
Completed experiment: bow, ngram=(1, 2), features=7000, Accuracy=0.7475
🏃 View run bow_ngram(1, 2)_features7000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1/runs/5435923ba4da477b903a2a192ccca8d3
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1
Completed experiment: bow, ngram=(1, 2), features=9000, Accuracy=0.7494
🏃 View run bow_ngram(1, 2)_features9000 at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlflow/#/experiments/1/runs/35de440a4a9a41269193b1c199214624
🧪 View experiment at: https://dagshub.com/iamdebasishdas123/YouTube-Mood-Tracker.mlf

### Experiments ML Algorithm

### Tuning Hyperparameter